> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTIin49w/5zM403525G-teHXho4SLHg/view?utm_content=DAGzTIin49w&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h29b78664e1)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3 pymupdf==1.26.7 jq==1.10.0 unstructured==0.18.26 openpyxl==3.1.5 networkx==3.6.1 msoffcrypto-tool==5.4.2 docx2txt==0.9 python-pptx==1.0.2 markdown==3.10 bilibili-api-python==17.4.1 youtube-transcript-api==1.2.3 pytube==15.0.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. 文档切分

## 2.1 简介
在文档内容被制作成向量数据库之前，我们需要先收集相关的内容：
- 从PDF、数据库、URLs等不同来源读取内容（目前一般只读取文本的相关内容）；
- 然后统一转成Documents格式，作为后续“切分—嵌入—检索”的输入。



## 2.2 载入方式

一般情况下，我们会通过 loader.load() 进行载入，其处理流程为：
- 遍历所有文件 / 页面
- 全部读完
- 全部变成 Document
- 一次性返回 list[Document]

虽然这种方法简单好理解，但是有一个很大的问题是非常吃内存。

因此对于数据量特别大的场景，langchain 中提供了 .lazy_load() 的载入方式，其不会立刻读数据，而是用完一个，再读下一个，这样的话整体的内存消耗就会被大大降低了。

### 2.2.1 载入 .txt 格式文件
最基础的 loader 格式，只需要安装 langchain_community 即可：

In [ ]:
# 最基础的 loader 格式，只需要安装 langchain_community 即可
from langchain_community.document_loaders import TextLoader
loader = TextLoader(file_path="./loaders_example/sample.txt", encoding="utf-8")
pages = loader.load()
print(pages) # 将文档里所有的内容都打印出来

### 2.2.2 载入网址内容
首先需要安装 beautifulsoup4 库:

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()
print(docs[0].page_content[:500]) # 将网页里面前500个字符打印出来

### 2.2.3 载入 .csv 文件
只需要 langchain-community 即可使用：

In [ ]:
# 只需要 langchain-community 即可 
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path="./loaders_example/sample.csv", encoding="utf-8")
pages = loader.load()
print(pages) 

### 2.2.4 载入 .pdf 格式文件
首先需要安装 pymupdf 库才能使用：

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(r"./loaders_example/sample.pdf")
pages = loader.load()
print(pages) 

### 2.2.5 载入 .json 格式文件
首先需要安装 jq 库才可使用：

In [ ]:
from langchain_community.document_loaders import JSONLoader
loader = JSONLoader(file_path="./loaders_example/sample.json", jq_schema=".", text_content=False) # "." 表示把整个 JSON 文件当做一个 Document
pages = loader.load()
print(pages) 

### 2.2.6 载入 .xlsx 格式文件
首先需要安装 unstructured 、msoffcrypto-tool、networkx、和 openpyxl 库：

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader
loader = UnstructuredExcelLoader(r"./loaders_example/sample.xlsx")
pages = loader.load()
print(pages) 

### 2.2.7 载入 .docx 格式文件
首先需要安装 docx2txt 库：

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader(r"./loaders_example/sample.docx")
pages = loader.load()
print(pages) 

### 2.2.8 载入 .ppt 文件
首先需要安装 unstructured, python-magic, python-pptx 库：

In [ ]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader
loader = UnstructuredPowerPointLoader(r"./loaders_example/sample.pptx")
pages = loader.load()
print(pages) 

### 2.2.9 载入 .html 格式文件
首先需要安装 beautifulsoup4 和 lxml 库：

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader
loader = BSHTMLLoader(r"./loaders_example/sample.html", open_encoding="utf-8")
pages = loader.load()
print(pages) 

### 2.2.10 载入 .md 格式文件
首先需要安装 unstructured 和 markdown 库：

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader(r"./loaders_example/sample.md")
pages = loader.load()
print(pages) 

### 2.2.11 载入 .ipynb 格式文件
不需要安装额外的库：

In [ ]:
from langchain_community.document_loaders import NotebookLoader
loader = NotebookLoader(r"./loaders_example/sample.ipynb", include_outputs=True, max_output_length=20, remove_newline=True)
pages = loader.load()
print(pages) 

### 2.2.12 载入 .xml 格式文件
无需安装额外的库：

In [ ]:
from langchain_community.document_loaders import UnstructuredXMLLoader
loader = UnstructuredXMLLoader(r"./loaders_example/sample.xml")
pages = loader.load()
print(pages) 

### 2.2.13 载入 bilibili 字幕文件

比如我们想要获取 B 站上某个视频的字幕作为我们文档数据的话，LangChain 中也有相关支持的 BiliBiliLoader 可以使用（需要安装 bilibili-api-python）。

我们需要打开并登录 B 站。然后打开开发者模式（F12）并找到应用程序。然后我们可以在筛选器里搜索三部分内容，并把对应的值保留下来：
- SESSDATA = `"<your sessdata>"`
- BUVID3 = `"<your buvids>"`
- BILI_JCT = `"<your bili_jct>"`

然后我们就可以创建载入器（loader）来对有字幕的视频进行提取了（不添加无法正确返回内容）：


In [ ]:
from langchain_community.document_loaders import BiliBiliLoader

loader = BiliBiliLoader(
  ["https://www.bilibili.com/video/BV1g84y1R7oE/"],
  sessdata = "<your sessdata>",
  buvid3 = "<your buvids>",
  bili_jct = "<your bili_jct>"
)

docs = loader.load()
print(docs)

### 2.2.14 载入 Youtube 字幕文件
类似的我们也可以获取 YouTube 上视频的字幕，这个相对没那么复杂，只需要使用 YoutubeLoader 并安装 pytube 和 youtube-transcript-api 库即可（需要外网才能够连接）：

In [ ]:
from langchain_community.document_loaders import YoutubeLoader

loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=QsYGlZkevEg",
    add_video_info=True,
    language=["en", "id"],
    translation="en",
)

docs = loader.load()
print(docs)